In [4]:
from pathlib import Path
from typing import Sequence, Tuple

import numpy as np
from scipy.integrate import ode
import torch
from tqdm.auto import tqdm

# -----------------------------------------------------------------------------
# Baseline model parameters (unchanged)
# -----------------------------------------------------------------------------
# Assume ``glucose_insulin_model``, ``params`` and ``x0`` are defined elsewhere
# and imported here.  They are *identical* to the ones you used previously.
from core_model import glucose_insulin_model, params, x0  # type: ignore

# -----------------------------------------------------------------------------
# Constants
# -----------------------------------------------------------------------------
BASE_D_RATE = 5 * (1000 / 180) / 40        # mmol·min⁻¹ for *three* meals per day
MEAL_DURATION = 40                         # min – fixed square‑wave width
B_LAPLACE     = 0.20                       # Laplace scale parameter (mmol/L)

T_END = 48 * 60                            # 48 h, minutes
DT     = 1                                 # 1‑min sampling
TS     = np.arange(0, T_END + DT, DT)
T      = TS.size

# -----------------------------------------------------------------------------
# 1. Meal time sampler (3–5 meals)
# -----------------------------------------------------------------------------

# Windows for potential meals (min after midnight)
_BASE_WINDOWS  = [(7*60, 11*60), (12*60, 14*60), (18*60, 21*60)]
_SNACK_WINDOWS = [(14*60, 17*60), (21*60, 23*60)]


def sample_meal_times(rng: np.random.Generator, n_meals: int) -> Tuple[int, ...]:
    """Return **sorted** start times (min after midnight) for *n_meals*.

    We always schedule breakfast, lunch and dinner in their normal windows.
    Extra meals (snacks) are drawn without replacement from the snack windows.
    """
    if n_meals < 3 or n_meals > 5:
        raise ValueError("n_meals must be 3, 4, or 5")

    windows = list(_BASE_WINDOWS)
    if n_meals > 3:
        extra = rng.choice(_SNACK_WINDOWS, size=n_meals - 3, replace=False)
        windows.extend(extra)

    times = [rng.integers(low, high) for low, high in windows]
    return tuple(sorted(times))

# -----------------------------------------------------------------------------
# 2. Glucose‑trace simulator with variable meal count / rate
# -----------------------------------------------------------------------------

def simulate_bg_trace(meal_times: Sequence[int], *, total_meals: int) -> np.ndarray:
    """Integrate the 6‑state model and return BG(t) for 48 h (shape ``(T,)``)."""

    per_meal_rate = BASE_D_RATE * 3 / total_meals  # scale so *area* is constant

    def D_of_t_local(t_min: float) -> float:
        return sum(per_meal_rate for m in meal_times if m <= t_min < m + MEAL_DURATION)

    def rhs(t: float, x: np.ndarray) -> np.ndarray:
        return glucose_insulin_model(t, x, params, D_of_t_local)

    solver = ode(rhs).set_integrator("dopri5")
    solver.set_initial_value(x0, 0.0)

    xs = np.empty((T, 6), dtype=np.float32)
    xs[0] = x0
    for k in range(1, T):
        xs[k] = solver.integrate(TS[k])
    return xs[:, 0]  # blood‑glucose only

# -----------------------------------------------------------------------------
# 3. Split generator (train / test)
# -----------------------------------------------------------------------------

def _make_split(N: int, *, is_train: bool, rng: np.random.Generator,
                b_laplace: float = B_LAPLACE):
    """Simulate *N* traces and return (bg_true, z_lap, meals, n_meals)."""

    bg_true = np.empty((N, T), dtype=np.float32)
    z_lap   = np.empty_like(bg_true)
    meals   = np.zeros((N, 5), dtype=np.int16)  # pad to 5 for convenience
    n_meals_arr = np.zeros(N, dtype=np.int8)

    for i in tqdm(range(N), desc="train" if is_train else "test"):
        n_meals = rng.integers(3, 6) if is_train else 3
        mt = sample_meal_times(rng, n_meals)

        bg = simulate_bg_trace(mt, total_meals=n_meals)

        bg_true[i] = bg
        z_lap[i]   = bg + rng.laplace(0.0, b_laplace, bg.shape)
        meals[i, :n_meals] = mt
        n_meals_arr[i]     = n_meals

    return dict(x=bg_true, z=z_lap, meal=meals, n=n_meals_arr)

# -----------------------------------------------------------------------------
# 4. Public helpers
# -----------------------------------------------------------------------------

def generate_dataset(*, n_train: int = 200, n_test: int = 200,
                     seed: int = 2025, out_dir: str = "data_knet_mixed",
                     b_laplace: float = B_LAPLACE) -> Path:
    """Generate Laplace‑noisy glucose datasets and save them under *out_dir*."""
    rng = np.random.default_rng(seed)

    out = Path(out_dir)
    out.mkdir(exist_ok=True)

    np.savez_compressed(out / "train.npz",
                        **_make_split(n_train, is_train=True,  rng=rng, b_laplace=b_laplace))
    np.savez_compressed(out / "test.npz",
                        **_make_split(n_test,  is_train=False, rng=rng, b_laplace=b_laplace))
    return out


class GlucoseDataset(torch.utils.data.Dataset):
    """Wraps (z_noisy, bg_true) into a PyTorch Dataset."""
    def __init__(self, z: np.ndarray, x_true: np.ndarray):
        self.z = torch.as_tensor(z, dtype=torch.float32)
        self.x = torch.as_tensor(x_true, dtype=torch.float32)

    def __len__(self):
        return len(self.z)

    def __getitem__(self, idx):
        return self.z[idx], self.x[idx]


def load_dataloaders(out_dir: str | Path, *, batch_size: int = 32):
    """Load *.npz files from *out_dir* and return train / test DataLoaders."""
    out_dir = Path(out_dir)
    train = np.load(out_dir / "train.npz")
    test  = np.load(out_dir / "test.npz")

    train_ds = GlucoseDataset(train["z"], train["x"])
    test_ds  = GlucoseDataset(test["z"],  test["x"])

    train_dl = torch.utils.data.DataLoader(train_ds, batch_size=batch_size,
                                           shuffle=True, drop_last=True)
    test_dl  = torch.utils.data.DataLoader(test_ds, batch_size=batch_size,
                                           shuffle=False)
    return train_dl, test_dl


ModuleNotFoundError: No module named 'core_model'